# Improved WiFi-HAR Model Training

This notebook implements key improvements to address the low accuracy issues:
1. Simplified model architecture for better convergence
2. Optimized hyperparameters
3. Better data splitting strategy
4. Simplified feature extraction
5. Improved training loop with validation monitoring

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import re
import warnings
warnings.filterwarnings('ignore')

# Set seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Device configuration
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using device: {device}")

Using device: mps


In [2]:
# IMPROVED CONFIGURATION
# Simplified and optimized hyperparameters

# Data parameters
DATA_PATH = Path('.')
WINDOW_SIZE = 64  # Increased for better temporal context
OVERLAP_RATIO = 0.5  # Reduced from 0.75 to prevent overfitting
STEP_SIZE = int(WINDOW_SIZE * (1 - OVERLAP_RATIO))

# Model parameters
BATCH_SIZE = 32  # Reduced for better gradient estimates
HIDDEN_SIZE = 128  # Simplified architecture
NUM_LAYERS = 2  # Reduced from 3
DROPOUT_RATE = 0.3  # Reduced dropout

# Training parameters
EPOCHS = 50
LEARNING_RATE = 0.001  # Increased from 5e-5 for better convergence
WEIGHT_DECAY = 1e-4  # Reduced regularization
PATIENCE = 10  # Early stopping patience

# Feature parameters
USE_STATISTICAL_FEATURES = True
USE_RAW_PACKETS = True

print(f"Configuration:")
print(f"  Window size: {WINDOW_SIZE}")
print(f"  Step size: {STEP_SIZE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Hidden size: {HIDDEN_SIZE}")

Configuration:
  Window size: 64
  Step size: 32
  Batch size: 32
  Learning rate: 0.001
  Hidden size: 128


In [3]:
def load_and_merge_csv_files():
    """Load and merge all CSV files with labels extracted from filenames."""
    csv_files = list(DATA_PATH.glob('wifisignal_data_*.csv'))
    
    if not csv_files:
        raise ValueError("No WiFi signal CSV files found!")
    
    print(f"Found {len(csv_files)} CSV files")
    
    df_list = []
    pattern = r'wifisignal_data_([a-zA-Z]+)_\d{8}_\d{6}\.csv'
    
    for file_path in csv_files:
        # Extract activity label from filename
        match = re.search(pattern, file_path.name)
        if match:
            activity_label = match.group(1)
            df = pd.read_csv(file_path)
            df['label'] = activity_label
            df['file_source'] = file_path.name  # Track source file
            df_list.append(df)
            print(f"  {activity_label}: {len(df)} samples from {file_path.name}")
        else:
            print(f"  Warning: Could not extract label from {file_path.name}")
    
    if not df_list:
        raise ValueError("No valid data files processed")
    
    combined_df = pd.concat(df_list, ignore_index=True)
    print(f"\nTotal combined data: {len(combined_df)} samples")
    print(f"Activities: {combined_df['label'].value_counts().to_dict()}")
    
    return combined_df

# Load data
raw_data = load_and_merge_csv_files()

Found 211 CSV files
  falling: 13881 samples from wifisignal_data_falling_20250528_175355.csv
  sitting: 16746 samples from wifisignal_data_sitting_20250528_151415.csv
  standing: 15402 samples from wifisignal_data_standing_20250529_152837.csv
  walking: 16583 samples from wifisignal_data_walking_20250528_165939.csv
  walking: 14459 samples from wifisignal_data_walking_20250529_160742.csv
  walking: 14775 samples from wifisignal_data_walking_20250529_161139.csv
  standing: 16514 samples from wifisignal_data_standing_20250529_151536.csv
  standing: 12280 samples from wifisignal_data_standing_20250528_142724.csv
  standing: 15264 samples from wifisignal_data_standing_20250528_155927.csv
  standing: 17507 samples from wifisignal_data_standing_20250528_143837.csv
  walking: 16220 samples from wifisignal_data_walking_20250528_171434.csv
  walking: 11115 samples from wifisignal_data_walking_20250529_164451.csv
  standing: 14845 samples from wifisignal_data_standing_20250528_174055.csv
  walk

In [4]:
def extract_simplified_features(df, window_size=WINDOW_SIZE, step_size=STEP_SIZE):
    """Extract simplified temporal features with better performance."""
    
    # Get packet columns
    packet_cols = [col for col in df.columns if col.startswith('pkt')]
    if not packet_cols:
        raise ValueError("No packet columns found")
    
    print(f"Extracting features from {len(packet_cols)} packet columns")
    print(f"Window size: {window_size}, Step size: {step_size}")
    
    features_list = []
    labels_list = []
    
    # Group by label for balanced windowing
    for label in df['label'].unique():
        label_data = df[df['label'] == label].copy().reset_index(drop=True)
        
        # Extract windows for this label
        for i in range(0, len(label_data) - window_size + 1, step_size):
            window = label_data.iloc[i:i+window_size]
            packet_data = window[packet_cols].values  # Shape: (window_size, n_packets)
            
            # Extract simplified features
            features = extract_window_features(packet_data)
            
            features_list.append(features)
            labels_list.append(label)
    
    # Create feature DataFrame
    feature_df = pd.DataFrame(features_list)
    feature_df['label'] = labels_list
    
    print(f"Extracted {len(feature_df)} feature windows")
    print(f"Feature shape: {feature_df.drop('label', axis=1).shape}")
    print(f"Label distribution: {feature_df['label'].value_counts().to_dict()}")
    
    return feature_df

def extract_window_features(packet_data):
    """Extract simplified but effective features from a window of packet data."""
    features = {}
    
    # Handle NaN values
    packet_data = np.nan_to_num(packet_data, nan=0.0)
    
    # 1. Basic statistical features across time
    features['mean_all'] = np.mean(packet_data)
    features['std_all'] = np.std(packet_data)
    features['max_all'] = np.max(packet_data)
    features['min_all'] = np.min(packet_data)
    features['range_all'] = features['max_all'] - features['min_all']
    
    # 2. Temporal dynamics (first-order differences)
    if packet_data.shape[0] > 1:
        temporal_diff = np.diff(packet_data, axis=0)
        features['temporal_mean'] = np.mean(temporal_diff)
        features['temporal_std'] = np.std(temporal_diff)
        features['temporal_max'] = np.max(np.abs(temporal_diff))
        features['temporal_energy'] = np.sum(temporal_diff**2)
    else:
        features['temporal_mean'] = 0.0
        features['temporal_std'] = 0.0
        features['temporal_max'] = 0.0
        features['temporal_energy'] = 0.0
    
    # 3. Per-packet statistics (sample 10 packets to avoid too many features)
    n_sample_packets = min(10, packet_data.shape[1])
    packet_indices = np.linspace(0, packet_data.shape[1]-1, n_sample_packets, dtype=int)
    
    for i, pkt_idx in enumerate(packet_indices):
        pkt_values = packet_data[:, pkt_idx]
        features[f'pkt_{i}_mean'] = np.mean(pkt_values)
        features[f'pkt_{i}_std'] = np.std(pkt_values)
        features[f'pkt_{i}_range'] = np.ptp(pkt_values)
    
    # 4. Cross-packet correlations (simplified)
    if packet_data.shape[1] > 1:
        # Sample 5 packets for correlation
        corr_indices = np.linspace(0, packet_data.shape[1]-1, min(5, packet_data.shape[1]), dtype=int)
        sampled_data = packet_data[:, corr_indices]
        
        try:
            corr_matrix = np.corrcoef(sampled_data.T)
            corr_matrix = np.nan_to_num(corr_matrix, nan=0.0)
            upper_tri = corr_matrix[np.triu_indices_from(corr_matrix, k=1)]
            
            if len(upper_tri) > 0:
                features['corr_mean'] = np.mean(upper_tri)
                features['corr_std'] = np.std(upper_tri)
                features['corr_max'] = np.max(np.abs(upper_tri))
            else:
                features['corr_mean'] = 0.0
                features['corr_std'] = 0.0
                features['corr_max'] = 0.0
        except:
            features['corr_mean'] = 0.0
            features['corr_std'] = 0.0
            features['corr_max'] = 0.0
    else:
        features['corr_mean'] = 0.0
        features['corr_std'] = 0.0
        features['corr_max'] = 0.0
    
    # 5. Activity indicators
    movement_intensity = np.sqrt(features['temporal_energy'])
    features['movement_intensity'] = movement_intensity
    features['is_static'] = 1.0 if movement_intensity < 0.1 else 0.0
    features['is_dynamic'] = 1.0 if movement_intensity > 1.0 else 0.0
    
    # Clean NaN/inf values
    for key, value in features.items():
        if np.isnan(value) or np.isinf(value):
            features[key] = 0.0
    
    return features

# Extract features
print("\nExtracting simplified temporal features...")
feature_data = extract_simplified_features(raw_data)


Extracting simplified temporal features...
Extracting features from 60 packet columns
Window size: 64, Step size: 32
Extracted 97691 feature windows
Feature shape: (97691, 45)
Label distribution: {'standing': 28164, 'walking': 26084, 'sitting': 14735, 'sleeping': 14592, 'falling': 14116}


In [5]:
def create_balanced_splits(df, test_size=0.2, val_size=0.2, random_state=42):
    """Create stratified train/val/test splits that avoid data leakage."""
    
    # Separate features and labels
    X = df.drop('label', axis=1)
    y = df['label']
    
    print(f"\nCreating balanced data splits:")
    print(f"Total samples: {len(df)}")
    print(f"Features: {X.shape[1]}")
    print(f"Classes: {y.nunique()}")
    
    # First split: train+val vs test
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=random_state
    )
    
    # Second split: train vs val
    val_size_adjusted = val_size / (1 - test_size)  # Adjust val_size for remaining data
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=val_size_adjusted, stratify=y_temp, random_state=random_state
    )
    
    print(f"\nSplit sizes:")
    print(f"  Training: {len(X_train)} ({len(X_train)/len(df)*100:.1f}%)")
    print(f"  Validation: {len(X_val)} ({len(X_val)/len(df)*100:.1f}%)")
    print(f"  Test: {len(X_test)} ({len(X_test)/len(df)*100:.1f}%)")
    
    # Check class balance
    print(f"\nClass distribution:")
    for split_name, split_y in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
        counts = split_y.value_counts()
        print(f"  {split_name}: {dict(counts)}")
    
    return X_train, X_val, X_test, y_train, y_val, y_test

# Create splits
X_train, X_val, X_test, y_train, y_val, y_test = create_balanced_splits(feature_data)


Creating balanced data splits:
Total samples: 97691
Features: 45
Classes: 5

Split sizes:
  Training: 58614 (60.0%)
  Validation: 19538 (20.0%)
  Test: 19539 (20.0%)

Class distribution:
  Train: {'standing': 16898, 'walking': 15650, 'sitting': 8841, 'sleeping': 8755, 'falling': 8470}
  Val: {'standing': 5633, 'walking': 5217, 'sitting': 2947, 'sleeping': 2918, 'falling': 2823}
  Test: {'standing': 5633, 'walking': 5217, 'sitting': 2947, 'sleeping': 2919, 'falling': 2823}


In [6]:
def preprocess_features(X_train, X_val, X_test, y_train, y_val, y_test):
    """Preprocess features and labels for training."""
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    # Encode labels
    label_encoder = LabelEncoder()
    y_train_encoded = label_encoder.fit_transform(y_train)
    y_val_encoded = label_encoder.transform(y_val)
    y_test_encoded = label_encoder.transform(y_test)
    
    print(f"\nPreprocessing completed:")
    print(f"  Feature scaling: mean={X_train_scaled.mean():.4f}, std={X_train_scaled.std():.4f}")
    print(f"  Label encoding: {dict(zip(label_encoder.classes_, range(len(label_encoder.classes_))))}")
    
    # Convert to tensors
    X_train_tensor = torch.FloatTensor(X_train_scaled)
    X_val_tensor = torch.FloatTensor(X_val_scaled)
    X_test_tensor = torch.FloatTensor(X_test_scaled)
    
    y_train_tensor = torch.LongTensor(y_train_encoded)
    y_val_tensor = torch.LongTensor(y_val_encoded)
    y_test_tensor = torch.LongTensor(y_test_encoded)
    
    # Create data loaders
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    return train_loader, val_loader, test_loader, label_encoder, scaler

# Preprocess data
train_loader, val_loader, test_loader, label_encoder, scaler = preprocess_features(
    X_train, X_val, X_test, y_train, y_val, y_test
)


Preprocessing completed:
  Feature scaling: mean=-0.0000, std=0.9661
  Label encoding: {'falling': 0, 'sitting': 1, 'sleeping': 2, 'standing': 3, 'walking': 4}


In [7]:
class ImprovedWiFiHARModel(nn.Module):
    """Simplified and more effective model architecture."""
    
    def __init__(self, input_size, num_classes, hidden_size=HIDDEN_SIZE, dropout_rate=DROPOUT_RATE):
        super(ImprovedWiFiHARModel, self).__init__()
        
        self.input_size = input_size
        self.num_classes = num_classes
        self.hidden_size = hidden_size
        
        # Simplified architecture with batch normalization
        self.network = nn.Sequential(
            # First hidden layer
            nn.Linear(input_size, hidden_size * 2),
            nn.BatchNorm1d(hidden_size * 2),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            # Second hidden layer
            nn.Linear(hidden_size * 2, hidden_size),
            nn.BatchNorm1d(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            # Third hidden layer
            nn.Linear(hidden_size, hidden_size // 2),
            nn.BatchNorm1d(hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout_rate * 0.5),
            
            # Output layer
            nn.Linear(hidden_size // 2, num_classes)
        )
        
        # Initialize weights
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        """Initialize weights for better training."""
        if isinstance(module, nn.Linear):
            nn.init.xavier_normal_(module.weight)
            if module.bias is not None:
                nn.init.constant_(module.bias, 0)
        elif isinstance(module, nn.BatchNorm1d):
            nn.init.constant_(module.weight, 1)
            nn.init.constant_(module.bias, 0)
    
    def forward(self, x):
        return self.network(x)

# Initialize model
input_size = X_train.shape[1]
num_classes = len(label_encoder.classes_)

model = ImprovedWiFiHARModel(input_size, num_classes)
model = model.to(device)

print(f"\nModel initialized:")
print(f"  Input size: {input_size}")
print(f"  Number of classes: {num_classes}")
print(f"  Hidden size: {HIDDEN_SIZE}")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


Model initialized:
  Input size: 45
  Number of classes: 5
  Hidden size: 128
  Total parameters: 54,149
  Trainable parameters: 54,149


In [8]:
# Setup training components

# Class weights for imbalanced data
class_weights = compute_class_weight(
    class_weight='balanced', 
    classes=np.unique(y_train_encoded), 
    y=y_train_encoded
)
class_weights = torch.FloatTensor(class_weights).to(device)

print(f"Class weights: {dict(zip(label_encoder.classes_, class_weights.cpu().numpy()))}")

# Loss function and optimizer
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(
    model.parameters(), 
    lr=LEARNING_RATE, 
    weight_decay=WEIGHT_DECAY
)

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='min', 
    factor=0.5, 
    patience=5, 
    verbose=True
)

print(f"\nTraining setup:")
print(f"  Loss function: CrossEntropyLoss with class weights")
print(f"  Optimizer: Adam (lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY})")
print(f"  Scheduler: ReduceLROnPlateau")

NameError: name 'y_train_encoded' is not defined

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    """Train model for one epoch."""
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        
        # Gradient clipping for stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = output.max(1)
        total += target.size(0)
        correct += predicted.eq(target).sum().item()
    
    avg_loss = total_loss / len(train_loader)
    accuracy = 100. * correct / total
    
    return avg_loss, accuracy

def validate_epoch(model, val_loader, criterion, device):
    """Validate model for one epoch."""
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)
            
            total_loss += loss.item()
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()
    
    avg_loss = total_loss / len(val_loader)
    accuracy = 100. * correct / total
    
    return avg_loss, accuracy

# Training loop
print(f"\nStarting training for {EPOCHS} epochs...")
print("=" * 80)

# Training history
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

best_val_acc = 0.0
best_val_loss = float('inf')
patience_counter = 0

for epoch in range(EPOCHS):
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validate
    val_loss, val_acc = validate_epoch(model, val_loader, criterion, device)
    
    # Update scheduler
    scheduler.step(val_loss)
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    # Print progress
    print(f"Epoch {epoch+1:2d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
    
    # Early stopping and model saving
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_val_loss = val_loss
        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'epoch': epoch,
            'val_acc': val_acc,
            'val_loss': val_loss,
            'label_encoder': label_encoder,
            'scaler': scaler
        }, 'best_improved_model.pt')
        patience_counter = 0
        print(f"  → New best validation accuracy: {val_acc:.2f}% (saved model)")
    else:
        patience_counter += 1
        print(f"  → No improvement ({patience_counter}/{PATIENCE})")
        
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

print("\n" + "=" * 80)
print(f"Training completed!")
print(f"Best validation accuracy: {best_val_acc:.2f}%")
print(f"Best validation loss: {best_val_loss:.4f}")

In [ ]:
# Load best model and evaluate on test set
checkpoint = torch.load('best_improved_model.pt')
model.load_state_dict(checkpoint['model_state_dict'])

# Test evaluation
test_loss, test_acc = validate_epoch(model, test_loader, criterion, device)

print(f"\nFinal Test Results:")
print(f"  Test Loss: {test_loss:.4f}")
print(f"  Test Accuracy: {test_acc:.2f}%")

# Generate detailed predictions for analysis
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        _, predicted = output.max(1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(target.cpu().numpy())

# Classification report
print(f"\nClassification Report:")
print(classification_report(
    all_targets, 
    all_preds, 
    target_names=label_encoder.classes_,
    digits=4
))

In [ ]:
# Plot training history
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

epochs_range = range(1, len(history['train_loss']) + 1)

# Loss curves
ax1.plot(epochs_range, history['train_loss'], 'b-', label='Training Loss', linewidth=2)
ax1.plot(epochs_range, history['val_loss'], 'r-', label='Validation Loss', linewidth=2)
ax1.set_title('Training and Validation Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy curves
ax2.plot(epochs_range, history['train_acc'], 'b-', label='Training Accuracy', linewidth=2)
ax2.plot(epochs_range, history['val_acc'], 'r-', label='Validation Accuracy', linewidth=2)
ax2.set_title('Training and Validation Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Confusion Matrix
cm = confusion_matrix(all_targets, all_preds)
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues',
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_,
    ax=ax3
)
ax3.set_title('Confusion Matrix (Test Set)')
ax3.set_xlabel('Predicted')
ax3.set_ylabel('Actual')

# Performance comparison
metrics_comparison = {
    'Original Model': [25.9],  # From AdjacentSamples.ipynb
    'Improved Model': [test_acc]
}

x_pos = np.arange(len(metrics_comparison))
accuracies = [metrics_comparison[model][0] for model in metrics_comparison]
colors = ['red', 'green']

bars = ax4.bar(x_pos, accuracies, color=colors, alpha=0.7)
ax4.set_title('Model Performance Comparison')
ax4.set_xlabel('Model')
ax4.set_ylabel('Test Accuracy (%)')
ax4.set_xticks(x_pos)
ax4.set_xticklabels(metrics_comparison.keys())
ax4.grid(True, alpha=0.3)

# Add value labels on bars
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height + 1,
             f'{acc:.1f}%', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Print improvement summary
improvement = test_acc - 25.9
print(f"\n" + "="*60)
print(f"IMPROVEMENT SUMMARY")
print(f"="*60)
print(f"Original model accuracy:  25.9%")
print(f"Improved model accuracy:  {test_acc:.1f}%")
print(f"Absolute improvement:     +{improvement:.1f} percentage points")
print(f"Relative improvement:     +{improvement/25.9*100:.1f}%")
print(f"="*60)